In [1]:
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import joblib

# add path to project root
import sys
PROJECT_ROOT = Path.cwd()
sys.path.append(str(PROJECT_ROOT))
if PROJECT_ROOT.name == "notebooks":
    sys.path.append(str(PROJECT_ROOT.parent))
    
    
from src.preprocess.gaming_text_preprocessor import GamingTextPreprocessor
from src.preprocess.gaming_text_labeler import GamingTextLabeler
from src.ensemble import ensemble
from src.model.game_model_collection import GamingModelCollection

# instantiate preprocessor and labeler
gaming_preprocessor = GamingTextPreprocessor()
gaming_labeler = GamingTextLabeler()

In [2]:
# config
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data' / 'processed_data').exists() and PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

CONFIG = {
    'seed': 7524,
    'data_dir': PROJECT_ROOT / 'data' / 'processed_data',
    'model_dir': PROJECT_ROOT / 'models',
    'text_col': 'message',
    'label_col': 'label'
}

np.random.seed(CONFIG['seed'])
seed = CONFIG['seed']
tc, lc = CONFIG['text_col'], CONFIG['label_col']

print('CONFIG loaded. Text column:', tc)


CONFIG loaded. Text column: message


In [3]:
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix

def _safe_confusion_rates(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        'FPR': fp / (fp + tn) if (fp + tn) > 0 else 0,
        'FNR': fn / (fn + tp) if (fn + tp) > 0 else 0,
        'TPR': tp / (tp + fn) if (tp + fn) > 0 else 0,
        'TNR': tn / (tn + fp) if (tn + fp) > 0 else 0,
    }
    
def score_func(y_true, y_pred, uncertain_label=-1, min_coverage=0.80):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    covered = y_pred != uncertain_label
    coverage = covered.mean()

    if coverage < min_coverage:
        return -np.inf

    return f1_score(
        y_true[covered],
        y_pred[covered],
        average="macro"
    )

def precision(y_true, y_pred):
    rates = _safe_confusion_rates(y_true, y_pred)
    return rates['TPR'] / (rates['TPR'] + rates['FPR']) if (rates['TPR'] + rates['FPR']) > 0 else 0

def _prediction_metrics(y_true, y_pred):
    return {
        'CV Macro F1': f1_score(y_true, y_pred, average='macro'),
        'CV Weighted F1': f1_score(y_true, y_pred, average='weighted'),
        'Accuracy': accuracy_score(y_true, y_pred),
        "Precision": precision(y_true, y_pred),
        **_safe_confusion_rates(y_true, y_pred),
    }

# Fit the Gaming Text Preprocessor 

In [4]:
# load WOT train data
d = CONFIG['data_dir']

wot_train = pd.read_parquet(d / 'wot' / 'x_train.parquet')
wot_train["data_source"] = "wot"

In [5]:
# load DOTA training data
dota_train = pd.read_parquet(d / 'dota' / 'x_train.parquet')
dota_train["data_source"] = "dota"

In [6]:
# combine datasets
train = pd.concat([wot_train, dota_train], ignore_index=True)

In [7]:
# train 
gaming_preprocessor.fit(train, text_column=tc)

In [8]:
wot_val = pd.read_parquet(d / 'wot' / 'x_validation.parquet')
wot_val["data_source"] = "wot"
dota_val = pd.read_parquet(d / 'dota' / 'x_validation.parquet')
dota_val["data_source"] = "dota"

val = pd.concat([wot_val, dota_val], ignore_index=True)

# Label Cleaning

In [9]:

# convert dota labels 
train = gaming_labeler.convert_three_class(
    train, 
    label_column=lc, 
    data_source_column='data_source',
    output_column='label_3class'
)
val = gaming_labeler.convert_three_class(
    val, 
    label_column=lc, 
    data_source_column='data_source',
    output_column='label_3class'
)


In [10]:
train["label_3class"].value_counts()

label_3class
0    40929
1     7492
2     5286
Name: count, dtype: int64

# Model Loading

In [11]:
# load all models from disk
model_dir = CONFIG['model_dir'] / "multi"
model_paths = list(model_dir.glob("gaming_all_*.joblib"))
models = [joblib.load(path) for path in model_paths]

models

[{'wot': {'pipes': {'Logistic Regression': Pipeline(steps=[('tfidf',
                     TfidfVectorizer(max_df=0.95, ngram_range=(1, 2),
                                     sublinear_tf=True, token_pattern=None,
                                     tokenizer=<function tokenize at 0x1184b6fc0>)),
                    ('clf',
                     LogisticRegression(C=3.0774851336928744,
                                        class_weight='balanced', max_iter=2000,
                                        penalty='l2', random_state=7524,
                                        solver='newton-cg',
                                        tol=0.00021806665627858358))]),
    'LinearSVC': Pipeline(steps=[('tfidf',
                     TfidfVectorizer(max_df=0.95, ngram_range=(1, 2),
                                     sublinear_tf=True, token_pattern=None,
                                     tokenizer=<function tokenize at 0x1184b6fc0>)),
                    ('clf',
                     Li

# Simple Ensemble

In [12]:
gaming_model_collection = GamingModelCollection(
    model_joblibs = model_paths,
)

In [13]:
gaming_model_collection.classifiers.keys()

dict_keys(['wot_Logistic_Regression', 'wot_LinearSVC', 'wot_Logistic_Regression_+_Calibrated(isotonic,_ensemble=False)', 'wot_LinearSVC_+_Calibrated(isotonic,_ensemble=False)', 'dota_Logistic_Regression', 'dota_LinearSVC', 'dota_Logistic_Regression_+_Calibrated(sigmoid,_ensemble=True)', 'dota_LinearSVC_+_Calibrated(sigmoid,_ensemble=True)', 'combined_Logistic_Regression', 'combined_LinearSVC', 'combined_Logistic_Regression_+_Calibrated(sigmoid,_ensemble=False)', 'combined_LinearSVC_+_Calibrated(isotonic,_ensemble=True)'])

In [14]:
# instantiate the ensemble with model paths
multiclass_ensemble = ensemble.Ensemble(
    model_collections=[gaming_model_collection]
)

In [15]:
# run simple majority voting on the training data
pred_train_multiclass_ensemble = multiclass_ensemble.predict_simple_majority(train[tc])

Predicting with SimpleMajority...
Input has 53707 samples.
wot_Logistic_Regression: (53707,)
wot_LinearSVC: (53707,)
wot_Logistic_Regression_+_Calibrated(isotonic,_ensemble=False): (53707,)
wot_LinearSVC_+_Calibrated(isotonic,_ensemble=False): (53707,)
dota_Logistic_Regression: (53707,)
dota_LinearSVC: (53707,)
dota_Logistic_Regression_+_Calibrated(sigmoid,_ensemble=True): (53707,)
dota_LinearSVC_+_Calibrated(sigmoid,_ensemble=True): (53707,)
combined_Logistic_Regression: (53707,)
combined_LinearSVC: (53707,)
combined_Logistic_Regression_+_Calibrated(sigmoid,_ensemble=False): (53707,)
combined_LinearSVC_+_Calibrated(isotonic,_ensemble=True): (53707,)
Collected predictions from 12 models.
Prediction matrix shape: (53707, 12)
Aggregating predictions from 12 models...


In [16]:
_prediction_metrics(train['label_3class'], pred_train_multiclass_ensemble)

{'CV Macro F1': 0.9160356887082272,
 'CV Weighted F1': 0.9589583920446748,
 'Accuracy': 0.9593721488818963,
 'Precision': np.float64(0.9918325428975101),
 'FPR': np.float64(0.007638462483114331),
 'FNR': np.float64(0.07240704500978473),
 'TPR': np.float64(0.9275929549902152),
 'TNR': np.float64(0.9923615375168857)}

# Fitted Ensemble

In [17]:
thresholds = np.arange(0.50, 0.99, 0.05)

In [18]:
combined_val_pred = multiclass_ensemble.fit_weighted_confidence_majority(X_val=train[tc], y_val=train['label_3class'], score_func=score_func, thresholds=thresholds)

Constructing confidence tensor...
Total models in ensemble: 12
Expected confidence shape: (n_samples=53707, n_classes=3)
Starting random search for n models: 12
Trial weights: (12,)
Weighted probabilities shape: (53707, 3)
Trial weights: (12,)
Weighted probabilities shape: (53707, 3)
Trial weights: (12,)
Weighted probabilities shape: (53707, 3)
Trial weights: (12,)
Weighted probabilities shape: (53707, 3)
Trial weights: (12,)
Weighted probabilities shape: (53707, 3)
Trial weights: (12,)
Weighted probabilities shape: (53707, 3)
Trial weights: (12,)
Weighted probabilities shape: (53707, 3)
Trial weights: (12,)
Weighted probabilities shape: (53707, 3)
Trial weights: (12,)
Weighted probabilities shape: (53707, 3)
Trial weights: (12,)
Weighted probabilities shape: (53707, 3)
Trial weights: (12,)
Weighted probabilities shape: (53707, 3)
Trial weights: (12,)
Weighted probabilities shape: (53707, 3)
Trial weights: (12,)
Weighted probabilities shape: (53707, 3)
Trial weights: (12,)
Weighted pro

In [19]:
combined_val_pred

(['wot_Logistic_Regression',
  'wot_LinearSVC',
  'wot_Logistic_Regression_+_Calibrated(isotonic,_ensemble=False)',
  'wot_LinearSVC_+_Calibrated(isotonic,_ensemble=False)',
  'dota_Logistic_Regression',
  'dota_LinearSVC',
  'dota_Logistic_Regression_+_Calibrated(sigmoid,_ensemble=True)',
  'dota_LinearSVC_+_Calibrated(sigmoid,_ensemble=True)',
  'combined_Logistic_Regression',
  'combined_LinearSVC',
  'combined_Logistic_Regression_+_Calibrated(sigmoid,_ensemble=False)',
  'combined_LinearSVC_+_Calibrated(isotonic,_ensemble=True)'],
 array([0.01500173, 0.06092781, 0.00230242, 0.04228979, 0.04470104,
        0.05293533, 0.09912261, 0.00050821, 0.35757238, 0.06847398,
        0.22296926, 0.03319543]),
 np.float64(0.7000000000000002),
 0.9547674107882674)

In [20]:
_prediction_metrics(train['label_3class'], pred_train_multiclass_ensemble)

{'CV Macro F1': 0.9160356887082272,
 'CV Weighted F1': 0.9589583920446748,
 'Accuracy': 0.9593721488818963,
 'Precision': np.float64(0.9918325428975101),
 'FPR': np.float64(0.007638462483114331),
 'FNR': np.float64(0.07240704500978473),
 'TPR': np.float64(0.9275929549902152),
 'TNR': np.float64(0.9923615375168857)}

# Hold-Out Performance

In [ ]:
combined_pred = multiclass_ensemble.predict_weighted_confidence_majority(X=val[tc])

Constructing confidence tensor...
Total models in ensemble: 12
Expected confidence shape: (n_samples=17056, n_classes=3)
Predicted labels shape: (17056,)


In [22]:
_prediction_metrics(val['label_3class'], combined_pred)

{'CV Macro F1': 0.7184363044993903,
 'CV Weighted F1': 0.86503704929297,
 'Accuracy': 0.8681988742964353,
 'Precision': np.float64(0.9492924101487568),
 'FPR': np.float64(0.0356107976208632),
 'FNR': np.float64(0.3333333333333333),
 'TPR': np.float64(0.6666666666666666),
 'TNR': np.float64(0.9643892023791368)}

# View Hist